In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, when, trim, lower, count, avg, desc, lit

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("storageName", "adlsproybuckd01")
dbutils.widgets.text("schema", "bronze")
dbutils.widgets.text("catalog", "catalog_pry_final")
dbutils.widgets.text("fecha_rutina", "")

In [0]:
container = dbutils.widgets.get("container")
storageName = dbutils.widgets.get("storageName")
schema = dbutils.widgets.get("schema")
catalog = dbutils.widgets.get("catalog")
fecha_rutina = dbutils.widgets.get("fecha_rutina")

In [0]:
def load_raw(file):
    return spark.read.csv(f"abfss://{container}@{storageName}.dfs.core.windows.net/{file}.csv", sep=';', header=True, inferSchema=True) \
                .withColumn("ingestion_date", current_timestamp())

def write_bronze(df,table):
    return df.write.mode("overwrite").insertInto(f"{catalog}.{schema}.{table}")

def proceso_main():
    df_ailes = load_raw("aisles")
    df_aisles_1 = (df_ailes.withColumnRenamed("aisle_id","pasillo_id")
                   .withColumnRenamed("aisle","pasillo")
                   )
    df_depart = load_raw("departments")
    df_depart_1 = (df_depart.withColumnRenamed("department_id","departamento_id")
                   .withColumnRenamed("department","departamento")
                   )
    df_orders = load_raw("orders")
    df_orders_1 = (df_orders.withColumnRenamed("order_id","orden_id")
                   .withColumnRenamed("order_number","nro_orden")
                   .withColumnRenamed("order_dow","orden_dow")
                   .withColumnRenamed("order_hour_of_day","hora_del_dia")
                   .withColumnRenamed("days_since_prior_order","dias_desde_orden")
                   )
    df_products = load_raw("products")
    df_products_1 = (df_products.withColumnRenamed("product_id","producto_id")
                   .withColumnRenamed("product_name","producto_nombre")
                   .withColumnRenamed("aisle_id","pasillo_id")
                   .withColumnRenamed("department_id","departamento_id")
                   .withColumn("departamento_id",when(col("departamento_id").isNull(),lit(21)).otherwise(col("departamento_id")))
                   .drop("_c4")
                   )
    df_order_prior = load_raw("order_products__prior")
    df_order_prior_1 = (df_order_prior.withColumnRenamed("order_id","orden_id")
                   .withColumnRenamed("product_id","producto_id")
                   .withColumnRenamed("add_to_cart_order","orden_agregado")
                   .withColumnRenamed("reordered","reordenado")
                   )
    write_bronze(df_aisles_1,"pasillo")
    write_bronze(df_depart_1,"departamentos")
    write_bronze(df_orders_1,"ordenes")
    write_bronze(df_products_1,"productos")
    write_bronze(df_order_prior_1,"ordenes_prior")


In [0]:
proceso_main()